# Step 3: Evaluation, Metrics & Error Analysis
### Project: Evaluating the Impact of RAG on Reducing LLM Hallucinations

In this notebook, we analyze the experiment results:
1. **Aggregate Benchmark Comparison** (Hallucination Rate, Faithfulness, Accuracy)
2. **Category-wise Breakdown**
3. **Visualizing Generated Plots**
4. **Qualitative Case Studies & Error Inspection**

In [ ]:
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
sys.path.append('..')

from src.config import RESULTS_FILE, PLOTS_DIR

df = pd.read_csv(RESULTS_FILE)
print(f"Loaded {len(df)} experiment rows.")
df.head(3)

## 1. Overall System Summary Metrics
Let's compute the overall hallucination rate and average faithfulness for each configuration.

In [ ]:
total = len(df)
summary = {
    "Baseline (No RAG)": {
        "Hallucination Rate": f"{(df['baseline_hallucinated'].sum() / total) * 100:.1f}%",
        "Avg Faithfulness": f"{df['baseline_faithfulness'].mean() * 100:.1f}%"
    },
    "RAG (Top-3 Strict)": {
        "Hallucination Rate": f"{(df['rag_k3_hallucinated'].sum() / total) * 100:.1f}%",
        "Avg Faithfulness": f"{df['rag_k3_faithfulness'].mean() * 100:.1f}%"
    },
    "RAG (Top-5 Strict)": {
        "Hallucination Rate": f"{(df['rag_k5_hallucinated'].sum() / total) * 100:.1f}%",
        "Avg Faithfulness": f"{df['rag_k5_faithfulness'].mean() * 100:.1f}%"
    },
    "RAG (Top-3 Loose)": {
        "Hallucination Rate": f"{(df['rag_loose_hallucinated'].sum() / total) * 100:.1f}%",
        "Avg Faithfulness": f"{df['rag_loose_faithfulness'].mean() * 100:.1f}%"
    }
}
pd.DataFrame(summary).T

## 2. Category-wise Hallucination Breakdown
Let's see how each category performed (Direct Fact vs Multi-Hop vs Out-of-Corpus vs Adversarial).

In [ ]:
cat_summary = []
for cat, group in df.groupby('category'):
    n = len(group)
    cat_summary.append({
        'Category': cat,
        'Questions': n,
        'Baseline Hallucinations': f"{(group['baseline_hallucinated'].sum()/n)*100:.1f}%",
        'RAG Top-3 Hallucinations': f"{(group['rag_k3_hallucinated'].sum()/n)*100:.1f}%"
    })
pd.DataFrame(cat_summary)

## 3. Visualizing Comparison Charts
Let's display the generated high-resolution comparison figures.

In [ ]:
import matplotlib.image as mpimg

fig, ax = plt.subplots(figsize=(10, 6))
img = mpimg.imread(str(PLOTS_DIR / 'hallucination_reduction.png'))
ax.imshow(img)
ax.axis('off')
plt.title("Hallucination Reduction by System Setting", fontsize=14, pad=10)
plt.show()

## 4. Inspecting Individual Failure Cases
Let's inspect questions where the baseline LLM made up facts vs how RAG handled them.

In [ ]:
samples = df[df['baseline_hallucinated'] == 1][['id', 'category', 'question', 'baseline_answer', 'rag_k3_answer', 'ground_truth']].head(5)
for _, r in samples.iterrows():
    print(f"[{r['id']}] Category: {r['category']}")
    print(f"Question: {r['question']}")
    print(f"Baseline: {r['baseline_answer']}")
    print(f"RAG Top-3: {r['rag_k3_answer']}")
    print(f"Ground Truth: {r['ground_truth']}")
    print("-" * 70)